# Read UMI End-Effector Positions

This notebook loads all `robot0_eef_pos` samples from a UMI-format Zarr dataset.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import zarr
import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message=r"Object at \\.DS_Store is not recognized as a component of a Zarr hierarchy\.",
)


In [ ]:
# Change this path if your dataset is elsewhere.
zarr_path = Path("pick_cube.zarr")
assert zarr_path.exists(), f"Zarr path not found: {zarr_path.resolve()}"

root = zarr.open(str(zarr_path), mode="r")
eef_pos = np.asarray(root["data"]["robot0_eef_pos"])
episode_ends = np.asarray(root["meta"]["episode_ends"])

print("eef_pos shape:", eef_pos.shape)
print("eef_pos dtype:", eef_pos.dtype)
print("num episodes:", len(episode_ends))
print("total timesteps:", int(episode_ends[-1]))


In [ ]:
# First 10 (x, y, z) end-effector positions
eef_pos[:10]


In [ ]:
# Plot x/y/z across all timesteps
t = np.arange(eef_pos.shape[0])
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
labels = ["x", "y", "z"]

for i, ax in enumerate(axes):
    ax.plot(t, eef_pos[:, i], linewidth=0.8)
    ax.set_ylabel(f"{labels[i]} (m)")

axes[-1].set_xlabel("timestep")
fig.suptitle("End Effector Position (all timesteps)")
fig.tight_layout()
plt.show()


In [ ]:
# Optional: 3D trajectory view
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.plot(eef_pos[:, 0], eef_pos[:, 1], eef_pos[:, 2], linewidth=0.5)
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_zlabel("z (m)")
ax.set_title("3D End Effector Trajectory")
plt.tight_layout()
plt.show()


## Time-Synced RGB + 3D Trajectory (Robust Renderer)

If previous widget code showed an empty frame, use this version.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# VS Code safe path: do NOT use `%matplotlib widget` unless ipympl is installed.
# This cell uses inline rendering and optionally ipywidgets slider.

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

root = zarr.open("pick_cube.zarr", mode="r")
eef_pos = np.asarray(root["data"]["robot0_eef_pos"])
rgb = root["data"]["camera0_rgb"]

start_t = 5000
end_t = 6500  # exclusive

def render_t(t):
    frame = np.asarray(rgb[t])
    traj = eef_pos[start_t:t+1]

    fig = plt.figure(figsize=(12, 5))
    ax_img = fig.add_subplot(1, 2, 1)
    ax_3d = fig.add_subplot(1, 2, 2, projection="3d")

    ax_img.imshow(frame)
    ax_img.set_title(f"camera0_rgb @ t={t}")
    ax_img.axis("off")

    ax_3d.plot(traj[:, 0], traj[:, 1], traj[:, 2], color="tab:blue", linewidth=1.2)
    ax_3d.scatter(*traj[-1], color="red", s=35, label="current")
    ax_3d.set_xlabel("x (m)")
    ax_3d.set_ylabel("y (m)")
    ax_3d.set_zlabel("z (m)")
    ax_3d.set_title("EEF trajectory")
    ax_3d.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

# Data sanity check
test_frame = np.asarray(rgb[start_t])
print("frame stats at start_t:", test_frame.shape, test_frame.dtype, test_frame.min(), test_frame.max())

if HAS_WIDGETS:
    slider = widgets.IntSlider(value=start_t, min=start_t, max=end_t-1, step=1, description="t")
    out = widgets.interactive_output(render_t, {"t": slider})
    display(slider, out)
else:
    print("ipywidgets not installed in this kernel. Install it or set `t` manually below.")
    t = start_t
    render_t(t)
